In [ ]:
#Kunihiro Iwasaki

In [ ]:
#14th june 2026

In [ ]:
#homework7-part4

In [1]:
!pip install --upgrade --quiet playwright
!pip install --upgrade --quiet beautifulsoup4
!pip install --upgrade --quiet lxml
!pip install --upgrade --quiet html5lib
!pip install --upgrade --quiet pandas
!pip install --upgrade --quiet nest_asyncio

In [2]:
# %pip install --quiet lxml html5lib beautifulsoup4 pandas 
# %pip install --quiet playwright 
# !playwright install-deps 
# !playwright install chromium firefox

In [9]:
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

import asyncio
import nest_asyncio

try:
    asyncio.get_running_loop()
    nest_asyncio.apply()
except RuntimeError:
    pass


In [13]:
!playwright install

169.2 MiB [                    ] 0% 0.0s169.2 MiB [                    ] 0% 116.7s169.2 MiB [                    ] 0% 463.0s169.2 MiB [                    ] 0% 310.5s169.2 MiB [                    ] 0% 268.6s169.2 MiB [                    ] 0% 421.9s169.2 MiB [                    ] 0% 397.0s169.2 MiB [                    ] 0% 363.6s169.2 MiB [                    ] 0% 337.8s169.2 MiB [                    ] 0% 311.0s169.2 MiB [                    ] 0% 329.8s169.2 MiB [                    ] 0% 312.7s169.2 MiB [                    ] 0% 376.0s169.2 MiB [                    ] 0% 340.2s169.2 MiB [                    ] 0% 325.8s169.2 MiB [                    ] 0% 319.6s169.2 MiB [                    ] 0% 315.9s169.2 MiB [                    ] 0% 312.5s169.2 MiB [                    ] 0% 310.8s169.2 MiB [                    ] 0% 340.9s169.2 MiB [                    ] 0% 316.4s169.2 MiB [                    ] 0% 305.4s169.2 MiB [                    ] 0% 295.1s169.2 MiB [                    ] 0% 

In [66]:
import os
import sys
from playwright.async_api import async_playwright   
from bs4 import BeautifulSoup

# 1. Google Colab環境かどうかの自動判定
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

# 2. ブラウザの起動準備
playwright = await async_playwright().start()

if IN_COLAB: 
    use_headless = True 
else: 
    use_headless = False

browser = await playwright.chromium.launch(headless=use_headless)

# 3. 新しいページ（タブ）を作成する
page = await browser.new_page()

# --------------------------------------------------
# 4. 指定したURLのサイトを開く
# --------------------------------------------------
url = 'https://www.tnwb.uscourts.gov/Search/Search.aspx?zoom_sort=0&zoom_xml=0&zoom_query=CAR&zoom_per_page=200&zoom_and=1&zoom_cat%5B%5D=-1'

rows = []

await page.goto(url)
await page.wait_for_load_state("networkidle")

html = await page.content()
doc = BeautifulSoup(html, 'html.parser')

In [67]:
#the URL to the case, the name of the case, the category (e.g. "Judge's Opinions), the additional details (terms match/size/pdf URL).
results = doc.find_all('div', class_=['result_block', 'result_altblock'])
results

[<div class="result_block">
 <div class="result_title"><b>1.</b> <a href="https://www.tnwb.uscourts.gov/Opinions/jdl/pdf/jdl20071024nn1.pdf#search=%22car%22" target="_blank">JDL: 04-24318 Jacquelline D. Black</a><span class="category"> [Judges' Opinions]</span></div>
 <div class="context">
 <b>...</b> the basis that the Debtor failed to prove that K's Auto had custody of the <span class="highlight">car</span> or knew of the whereabouts of the <span class="highlight">car</span>. This adversary proceeding was administratively <b>...</b></div>
 <div class="infoline">Terms matched:  1  -  102k  -  URL: https://www.tnwb.uscourts.gov/Opinions/jdl/pdf/jdl20071024nn1.pdf</div>
 </div>,
 <div class="result_altblock">
 <div class="result_title"><b>2.</b> <a href="https://www.tnwb.uscourts.gov/Opinions/whb/pdf/whb19950809xn1.pdf#search=%22car%22" target="_blank">WHB: 95-26401 Mary Lucy Cooper</a><span class="category"> [Judges' Opinions]</span></div>
 <div class="context">
 <b>...</b> MARY LUCY C

In [68]:
rows = []

for result in results:
    row = {}
    result_title = result.find('div', attrs={'class':'result_title'}).get_text(strip=True)
    url = result.find('a').get('href', None)
    category = result.find(class_='category').get_text(strip=True)
    detail = result.find('div', attrs={'class':'infoline'}).get_text(strip=True)

    row['result_title'] = result_title
    row['url'] = url
    row['category'] = category
    row['detail'] = detail

    rows.append(row)
print(rows)

[{'result_title': "1.JDL: 04-24318 Jacquelline D. Black[Judges' Opinions]", 'url': 'https://www.tnwb.uscourts.gov/Opinions/jdl/pdf/jdl20071024nn1.pdf#search=%22car%22', 'category': "[Judges' Opinions]", 'detail': 'Terms matched:  1\xa0 - \xa0102k\xa0 - \xa0URL: https://www.tnwb.uscourts.gov/Opinions/jdl/pdf/jdl20071024nn1.pdf'}, {'result_title': "2.WHB: 95-26401 Mary Lucy Cooper[Judges' Opinions]", 'url': 'https://www.tnwb.uscourts.gov/Opinions/whb/pdf/whb19950809xn1.pdf#search=%22car%22', 'category': "[Judges' Opinions]", 'detail': 'Terms matched:  1\xa0 - \xa027k\xa0 - \xa0URL: https://www.tnwb.uscourts.gov/Opinions/whb/pdf/whb19950809xn1.pdf'}, {'result_title': "3.GHB: 97-12368 Billy G. Woffard[Judges' Opinions]", 'url': 'https://www.tnwb.uscourts.gov/Opinions/ghb/pdf/ghb19980812xn1.pdf#search=%22car%22', 'category': "[Judges' Opinions]", 'detail': 'Terms matched:  1\xa0 - \xa071k\xa0 - \xa0URL: https://www.tnwb.uscourts.gov/Opinions/ghb/pdf/ghb19980812xn1.pdf'}, {'result_title': "4

In [69]:
import pandas as pd
df = pd.DataFrame(rows)
print(df)
df.to_csv('homework7__part4.csv')

                                          result_title  \
0    1.JDL: 04-24318 Jacquelline D. Black[Judges' O...   
1    2.WHB: 95-26401 Mary Lucy Cooper[Judges' Opini...   
2    3.GHB: 97-12368 Billy G. Woffard[Judges' Opini...   
3    4.JDL: 97-30580 Mary Chrlis Hurst[Judges' Opin...   
4    5.MRH: 20-20967 Jacob Braxton Herring 20-00094...   
..                                                 ...   
131  132.WHB: 90-20283 The Julien Company[Judges' O...   
132  133.DEB: 24-22894 Julianka Michelle Jackson[Ju...   
133  134.WHB: 03-41991 Earline Jones, 04-22034 Will...   
134                     135.TNWB :: Judge Latta[Other]   
135  136.JLC: 14-12505 Joe M. & Angela J. Smith[Jud...   

                                                   url            category  \
0    https://www.tnwb.uscourts.gov/Opinions/jdl/pdf...  [Judges' Opinions]   
1    https://www.tnwb.uscourts.gov/Opinions/whb/pdf...  [Judges' Opinions]   
2    https://www.tnwb.uscourts.gov/Opinions/ghb/pdf...  [Judges' Opin

In [ ]:
for page_num in range(1, 11):
    print(f"--- {page_num} ページ目を処理中 ---")

    target_url = f"{base_url}&zoom_page={page_num}"

    await page.goto(target_url)
    await page.wait_for_load_state("networkidle")

    html = await page.content()
    doc = BeautifulSoup(html, 'html.parser')